## seed_dim_date
Populates `silver.dim_date` at **daily grain** (one row per calendar day) so it can be marked as a Power BI Date Table — mark-as-date-table requires a contiguous, gap-free date column. `is_month_end` / `is_quarter_end` flag the month-end / Mar-Jun-Sep-Dec month-end days, so the single daily grain still serves the monthly facts and the quarterly FHFA fact (which join on those `date_key` values). Range starts 1947 (CPI history; also covers the earliest fact key, FHFA 1975-Q3) and runs to 2031 (~30.7k rows). Full rebuild via `INSERT OVERWRITE` on `date_key` (deterministic yyyymmdd).</cell id="cell-0">

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injects SILVER, F, spark. date_key = yyyymmdd of full_date.
# Daily grain (one row per calendar day) so dim_date can be marked as a Power BI Date Table.
# Floor stays 1947: it must cover the earliest fact date_key (FHFA 1975-Q3) and CPI reaches 1947.
# Bounds are FULL calendar years (Jan 1 .. Dec 31) — mark-as-date-table wants whole years, and a
# daily END of 12-01 would truncate the final December (the old month-end seed hid this via last_day).
START = "1947-01-01"
END   = "2031-12-31"

cal = spark.sql(
    f"SELECT explode(sequence(to_date('{START}'), to_date('{END}'), interval 1 day)) AS full_date"
)
# Column order matches the dim_date DDL (so INSERT OVERWRITE ... SELECT * lines up positionally).
dim = cal.select(
    (F.year("full_date") * 10000 + F.month("full_date") * 100
     + F.dayofmonth("full_date")).cast("int").alias("date_key"),
    F.col("full_date"),
    F.year("full_date").alias("year"),
    F.quarter("full_date").alias("quarter"),
    F.month("full_date").alias("month"),
    F.trunc("full_date", "MM").alias("month_start"),
    F.trunc("full_date", "quarter").alias("quarter_start"),
    # is_month_end: this day is its month's last day. is_quarter_end: also a Mar/Jun/Sep/Dec end.
    (F.col("full_date") == F.last_day("full_date")).alias("is_month_end"),
    ((F.col("full_date") == F.last_day("full_date"))
     & F.month("full_date").isin(3, 6, 9, 12)).alias("is_quarter_end"),
    F.current_timestamp().alias("inserted_ts"),
    F.current_timestamp().alias("updated_ts"),
)
dim.createOrReplaceTempView("dim_date_staging")

# Full rebuild: the daily grain supersedes the prior month-end rows. INSERT OVERWRITE preserves
# the table's schema/PK/COMMENTs (DDL owns those) and avoids a half-migrated mix of grains.
spark.sql(f"INSERT OVERWRITE TABLE {SILVER}.dim_date SELECT * FROM dim_date_staging")
row_count = spark.table(f"{SILVER}.dim_date").count()
print(f"seed_dim_date: dim_date rows = {row_count:,}")